Save base Logit, KNN, SVC, LogisticRegression, CatBoost, XGboost with pipeline. Sedimentation_Stk feature was used!

Based on `research_14_SMOTE_influence_on_metrics.ipynb` it was decided to use SMOTE for all models, except Logit!

# Import libraries

In [1]:
# Changes to all modules will automatically be applied when any cell runs. 
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

import joblib

from statsmodels.api import Logit
from statsmodels.api import Logit # type: ignore
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier

from sklearn.metrics import (
    make_scorer,
    f1_score,
)

import optuna
from functools import partial

from pathlib import Path
import sys

sys.path.append(
    str(Path('..', 'utils_functionality', 'split_utils'))
)
sys.path.append(
    str(Path('..', 'utils_functionality', 'models'))
)

from split_tools import get_train_test
from modelling4_utils import (
    # _prepare_df,
    MLPipeline,
    _create_pipeline,
    StatsModelsEstimator,
    OptunaOptimizer,
    smote_objective,
    pure_smote_objective,
    GridSearchOptimizer,
    RANDOM_STATE,
)

seed = RANDOM_STATE

In [3]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Settings

In [4]:
model_postfix = 'optuna-opt-mean-smote_default-model'
add_smote = True
is_smotenc = True
smote_params = {
    'categorical_features': (
        'wettability',
        'inclination',
    ),
}
scoring_metrics = {"f1_macro": make_scorer(f1_score, average="macro"),}
step_scoring_average = "mean"
n_trials = 400

save_model_and_metrics = True
metrics_file = 'metrics_modelling4_smotenc_mean_opt.xlsx'

## Optimization functions
Optuna will not be used further, except first try, it is enough to conduct grid search

In [5]:
def optimize_smote_optuna(
    target:str,
    estimator:object,
    estimator_params:dict,
    n_trials:int=n_trials,
    step_scoring_average:str=step_scoring_average,
    model_postfix:str=model_postfix,
    add_smote:bool=add_smote,
    is_smotenc:bool=is_smotenc,
    smote_params:dict=smote_params,
    scoring_metrics:dict=scoring_metrics,
    save_model_and_metrics:bool=save_model_and_metrics,
    metrics_file:str=metrics_file,
    seed:int=seed,
):
    """
    Optimize the SMOTE parameters for a given estimator.

    Args:
        target: The target variable for the model.
        estimator: The machine learning estimator to be optimized.
        estimator_params: Parameters for the estimator.
        n_trials: The number of trials for optimization. Defaults to n_trials.
        model_postfix: A postfix to append to the model name. Defaults to model_postfix.
        add_smote: Whether to add SMOTE to the pipeline. Defaults to add_smote.
        is_smotenc: Whether to use SMOTENC for categorical features. Defaults to is_smotenc.
        smote_params: Parameters for SMOTE. Defaults to smote_params.
        scoring_metrics: Metrics for scoring the model. Defaults to scoring_metrics.
        save_model_and_metrics: Whether to save the model and metrics. Defaults to save_model_and_metrics.
    """
    
    smote_params = smote_params.copy()
    
    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params=estimator_params,
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=smote_params,
        scoring_metrics=scoring_metrics,
        step_scoring_average=step_scoring_average,
    )

    opt = OptunaOptimizer(
        objective=partial(smote_objective, ml_pipe=ml_pipe),
        study_name="smote_study",
        direction="maximize",
        seed=seed,
    )

    opt.optimize(n_trials=n_trials)

    best_params = opt.study.best_params
    print('best_params')
    display(best_params)

    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params=estimator_params,
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params={**smote_params, **best_params},
        metrics_file=metrics_file,
    )
    
    ml_pipe.run(
        save_model_and_metrics=save_model_and_metrics,
    )

# Logistic Regression

In [6]:
estimator = LogisticRegression
target = 'splashing'
estimator_params = {
    "fit_intercept": False,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-16 00:47:53,648] A new study created in memory with name: smote_study
[I 2025-04-16 00:47:53,992] Trial 0 finished with value: 0.8068955952248212 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.8068955952248212.
[I 2025-04-16 00:47:54,096] Trial 1 finished with value: 0.8093303573473086 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 1 with value: 0.8093303573473086.
[I 2025-04-16 00:47:54,202] Trial 2 finished with value: 0.8074201035939587 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 1 with value: 0.8093303573473086.
[I 2025-04-16 00:47:54,308] Trial 3 finished with value: 0.8087537607513812 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 1 with value: 0.8093303573473086.
[I 2025-04-16 00:47:54,411] Trial 4 finished with value: 0.8068384611533034 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "LogisticRegression"


,0
target,splashing
model,LogisticRegression_splashing_smotenc_optuna-op...
holdout_test_f1_macro,0.777184
holdout_test_accuracy_balanced,0.770833
holdout_test_roc_auc,0.885802
holdout_test_f1,0.848485
holdout_test_accuracy,0.8
cv_test_f1_macro_median,0.763393
cv_test_accuracy_balanced_median,0.77709
cv_test_roc_auc_median,0.874613


Model saved in ../results/models_modelling4/LogisticRegression_splashing_smotenc_optuna-opt-mean-smote_default-model


In [7]:
estimator = LogisticRegression
target = 'splashing'
estimator_params = {
    "fit_intercept": False,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-16 00:48:39,863] A new study created in memory with name: smote_study
[I 2025-04-16 00:48:39,965] Trial 0 finished with value: 0.8068955952248212 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.8068955952248212.
[I 2025-04-16 00:48:40,065] Trial 1 finished with value: 0.8093303573473086 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 1 with value: 0.8093303573473086.
[I 2025-04-16 00:48:40,162] Trial 2 finished with value: 0.8074201035939587 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 1 with value: 0.8093303573473086.
[I 2025-04-16 00:48:40,262] Trial 3 finished with value: 0.8087537607513812 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 1 with value: 0.8093303573473086.
[I 2025-04-16 00:48:40,363] Trial 4 finished with value: 0.8068384611533034 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "LogisticRegression"


,0
target,splashing
model,LogisticRegression_splashing_smotenc_optuna-op...
holdout_test_f1_macro,0.777184
holdout_test_accuracy_balanced,0.770833
holdout_test_roc_auc,0.885802
holdout_test_f1,0.848485
holdout_test_accuracy,0.8
cv_test_f1_macro_median,0.763393
cv_test_accuracy_balanced_median,0.77709
cv_test_roc_auc_median,0.874613


Model saved in ../results/models_modelling4/LogisticRegression_splashing_smotenc_optuna-opt-mean-smote_default-model


Full grid search can find better solution, than Optuna with small number of iterations - like 40! It appears in some Random_seed

In [8]:
estimator = LogisticRegression
target = 'no_fragmentation'
estimator_params = {
    "fit_intercept": False,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-16 00:49:23,692] A new study created in memory with name: smote_study
[I 2025-04-16 00:49:23,786] Trial 0 finished with value: 0.7893690960867744 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.7893690960867744.
[I 2025-04-16 00:49:23,876] Trial 1 finished with value: 0.7961089219778902 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 1 with value: 0.7961089219778902.
[I 2025-04-16 00:49:23,964] Trial 2 finished with value: 0.7893690960867744 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 1 with value: 0.7961089219778902.
[I 2025-04-16 00:49:24,055] Trial 3 finished with value: 0.7917448628083663 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 1 with value: 0.7961089219778902.
[I 2025-04-16 00:49:24,152] Trial 4 finished with value: 0.7927358172501057 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "LogisticRegression"


,0
target,no_fragmentation
model,LogisticRegression_no_fragmentation_smotenc_op...
holdout_test_f1_macro,0.785539
holdout_test_accuracy_balanced,0.825
holdout_test_roc_auc,0.952727
holdout_test_f1,0.708333
holdout_test_accuracy,0.813333
cv_test_f1_macro_median,0.826797
cv_test_accuracy_balanced_median,0.869231
cv_test_roc_auc_median,0.945055


Model saved in ../results/models_modelling4/LogisticRegression_no_fragmentation_smotenc_optuna-opt-mean-smote_default-model


# KNClassifier

In [9]:
estimator = KNeighborsClassifier
target = 'splashing'
estimator_params = {}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-16 00:50:03,040] A new study created in memory with name: smote_study
[I 2025-04-16 00:50:03,207] Trial 0 finished with value: 0.833176217119686 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.833176217119686.
[I 2025-04-16 00:50:03,332] Trial 1 finished with value: 0.8366591229825184 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 1 with value: 0.8366591229825184.
[I 2025-04-16 00:50:03,460] Trial 2 finished with value: 0.8344734678470631 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 1 with value: 0.8366591229825184.
[I 2025-04-16 00:50:03,586] Trial 3 finished with value: 0.8327111603202334 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 1 with value: 0.8366591229825184.
[I 2025-04-16 00:50:03,716] Trial 4 finished with value: 0.8295692621139823 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best i

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "KNeighborsClassifier"


,0
target,splashing
model,KNeighborsClassifier_splashing_smotenc_optuna-...
holdout_test_f1_macro,0.836601
holdout_test_accuracy_balanced,0.828704
holdout_test_roc_auc,0.883488
holdout_test_f1,0.888889
holdout_test_accuracy,0.853333
cv_test_f1_macro_median,0.844575
cv_test_accuracy_balanced_median,0.870743
cv_test_roc_auc_median,0.930075


Model saved in ../results/models_modelling4/KNeighborsClassifier_splashing_smotenc_optuna-opt-mean-smote_default-model


In [10]:
estimator = KNeighborsClassifier
target = 'no_fragmentation'
estimator_params = {}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-16 00:50:58,286] A new study created in memory with name: smote_study
[I 2025-04-16 00:50:58,414] Trial 0 finished with value: 0.8121592476536206 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.8121592476536206.
[I 2025-04-16 00:50:58,537] Trial 1 finished with value: 0.8212452005407724 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 1 with value: 0.8212452005407724.
[I 2025-04-16 00:50:58,653] Trial 2 finished with value: 0.8239651236730726 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 2 with value: 0.8239651236730726.
[I 2025-04-16 00:50:58,777] Trial 3 finished with value: 0.8191221072565593 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 2 with value: 0.8239651236730726.
[I 2025-04-16 00:50:58,916] Trial 4 finished with value: 0.8034919521746071 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "KNeighborsClassifier"


,0
target,no_fragmentation
model,KNeighborsClassifier_no_fragmentation_smotenc_...
holdout_test_f1_macro,0.834656
holdout_test_accuracy_balanced,0.845455
holdout_test_roc_auc,0.945455
holdout_test_f1,0.761905
holdout_test_accuracy,0.866667
cv_test_f1_macro_median,0.847115
cv_test_accuracy_balanced_median,0.848718
cv_test_roc_auc_median,0.931319


Model saved in ../results/models_modelling4/KNeighborsClassifier_no_fragmentation_smotenc_optuna-opt-mean-smote_default-model


# SVC

In [11]:
estimator = SVC
target = 'splashing'
estimator_params = {
    'probability': True,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-16 00:51:49,207] A new study created in memory with name: smote_study
[I 2025-04-16 00:51:49,354] Trial 0 finished with value: 0.8620559168302458 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.8620559168302458.
[I 2025-04-16 00:51:49,490] Trial 1 finished with value: 0.8606024698993406 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 0 with value: 0.8620559168302458.
[I 2025-04-16 00:51:49,618] Trial 2 finished with value: 0.8588694020675373 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 0 with value: 0.8620559168302458.
[I 2025-04-16 00:51:49,761] Trial 3 finished with value: 0.8500846768064889 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 0 with value: 0.8620559168302458.
[I 2025-04-16 00:51:49,903] Trial 4 finished with value: 0.8642617849079628 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "SVC"


,0
target,splashing
model,SVC_splashing_smotenc_optuna-opt-mean-smote_de...
holdout_test_f1_macro,0.810348
holdout_test_accuracy_balanced,0.80787
holdout_test_roc_auc,0.893519
holdout_test_f1,0.865979
holdout_test_accuracy,0.826667
cv_test_f1_macro_median,0.844575
cv_test_accuracy_balanced_median,0.870743
cv_test_roc_auc_median,0.934985


Model saved in ../results/models_modelling4/SVC_splashing_smotenc_optuna-opt-mean-smote_default-model


In [12]:
estimator = SVC
target = 'no_fragmentation'
estimator_params = {
    'probability': True,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-16 00:52:49,663] A new study created in memory with name: smote_study
[I 2025-04-16 00:52:49,810] Trial 0 finished with value: 0.832140607661619 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.832140607661619.
[I 2025-04-16 00:52:49,945] Trial 1 finished with value: 0.8346658965165396 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 1 with value: 0.8346658965165396.
[I 2025-04-16 00:52:50,069] Trial 2 finished with value: 0.8432794192989314 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 2 with value: 0.8432794192989314.
[I 2025-04-16 00:52:50,210] Trial 3 finished with value: 0.836698142857439 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 2 with value: 0.8432794192989314.
[I 2025-04-16 00:52:50,343] Trial 4 finished with value: 0.8208641644069442 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best is

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "SVC"


,0
target,no_fragmentation
model,SVC_no_fragmentation_smotenc_optuna-opt-mean-s...
holdout_test_f1_macro,0.843227
holdout_test_accuracy_balanced,0.877273
holdout_test_roc_auc,0.952727
holdout_test_f1,0.782609
holdout_test_accuracy,0.866667
cv_test_f1_macro_median,0.850704
cv_test_accuracy_balanced_median,0.89011
cv_test_roc_auc_median,0.961538


Model saved in ../results/models_modelling4/SVC_no_fragmentation_smotenc_optuna-opt-mean-smote_default-model


# CatBoost

In [13]:
estimator = CatBoostClassifier
target = 'splashing'
estimator_params = {
    'verbose': False,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-16 00:53:44,548] A new study created in memory with name: smote_study
[I 2025-04-16 00:53:49,297] Trial 0 finished with value: 0.8800298923107119 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.8800298923107119.
[I 2025-04-16 00:53:53,786] Trial 1 finished with value: 0.8835594406678464 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 1 with value: 0.8835594406678464.
[I 2025-04-16 00:53:58,011] Trial 2 finished with value: 0.8900278404468924 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 2 with value: 0.8900278404468924.
[I 2025-04-16 00:54:02,517] Trial 3 finished with value: 0.8755977547957539 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 2 with value: 0.8900278404468924.
[I 2025-04-16 00:54:06,972] Trial 4 finished with value: 0.8866947989672991 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "CatBoostClassifier"


,0
target,splashing
model,CatBoostClassifier_splashing_smotenc_optuna-op...
holdout_test_f1_macro,0.85
holdout_test_accuracy_balanced,0.83912
holdout_test_roc_auc,0.958333
holdout_test_f1,0.9
holdout_test_accuracy,0.866667
cv_test_f1_macro_median,0.898584
cv_test_accuracy_balanced_median,0.903251
cv_test_roc_auc_median,0.94582


Model saved in ../results/models_modelling4/CatBoostClassifier_splashing_smotenc_optuna-opt-mean-smote_default-model


In [14]:
estimator = CatBoostClassifier
target = 'no_fragmentation'
estimator_params = {
    'verbose': False,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-16 01:22:16,385] A new study created in memory with name: smote_study
[I 2025-04-16 01:22:21,201] Trial 0 finished with value: 0.8481424870661538 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.8481424870661538.
[I 2025-04-16 01:22:25,667] Trial 1 finished with value: 0.8500371673615249 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 1 with value: 0.8500371673615249.
[I 2025-04-16 01:22:29,975] Trial 2 finished with value: 0.8504523830569923 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 2 with value: 0.8504523830569923.
[I 2025-04-16 01:22:34,602] Trial 3 finished with value: 0.8522647763143156 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 3 with value: 0.8522647763143156.
[I 2025-04-16 01:22:39,180] Trial 4 finished with value: 0.8507752383134809 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "CatBoostClassifier"


,0
target,no_fragmentation
model,CatBoostClassifier_no_fragmentation_smotenc_op...
holdout_test_f1_macro,0.916089
holdout_test_accuracy_balanced,0.922727
holdout_test_roc_auc,0.987273
holdout_test_f1,0.878049
holdout_test_accuracy,0.933333
cv_test_f1_macro_median,0.90293
cv_test_accuracy_balanced_median,0.892857
cv_test_roc_auc_median,0.974359


Model saved in ../results/models_modelling4/CatBoostClassifier_no_fragmentation_smotenc_optuna-opt-mean-smote_default-model


# XGBoost

In [15]:
estimator = XGBClassifier
target = 'splashing'
estimator_params = {}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-16 01:51:53,440] A new study created in memory with name: smote_study
[I 2025-04-16 01:51:55,180] Trial 0 finished with value: 0.8784203918741195 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.8784203918741195.
[I 2025-04-16 01:51:56,987] Trial 1 finished with value: 0.8779950228452413 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 0 with value: 0.8784203918741195.
[I 2025-04-16 01:51:58,709] Trial 2 finished with value: 0.8554353174694594 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 0 with value: 0.8784203918741195.
[I 2025-04-16 01:52:00,423] Trial 3 finished with value: 0.8885832532610579 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 3 with value: 0.8885832532610579.
[I 2025-04-16 01:52:02,162] Trial 4 finished with value: 0.877905444109132 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best 

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "XGBClassifier"


,0
target,splashing
model,XGBClassifier_splashing_smotenc_optuna-opt-mea...
holdout_test_f1_macro,0.88
holdout_test_accuracy_balanced,0.868056
holdout_test_roc_auc,0.939429
holdout_test_f1,0.92
holdout_test_accuracy,0.893333
cv_test_f1_macro_median,0.831746
cv_test_accuracy_balanced_median,0.844444
cv_test_roc_auc_median,0.950464


Model saved in ../results/models_modelling4/XGBClassifier_splashing_smotenc_optuna-opt-mean-smote_default-model


In [16]:
estimator = XGBClassifier
target = 'no_fragmentation'
estimator_params = {}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-16 02:03:21,035] A new study created in memory with name: smote_study
[I 2025-04-16 02:03:22,650] Trial 0 finished with value: 0.847831651690411 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.847831651690411.
[I 2025-04-16 02:03:24,217] Trial 1 finished with value: 0.8286555540633209 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 0 with value: 0.847831651690411.
[I 2025-04-16 02:03:25,738] Trial 2 finished with value: 0.8403178155590403 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 0 with value: 0.847831651690411.
[I 2025-04-16 02:03:27,314] Trial 3 finished with value: 0.834343507224908 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 0 with value: 0.847831651690411.
[I 2025-04-16 02:03:28,882] Trial 4 finished with value: 0.832581533017551 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best is tri

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "XGBClassifier"


,0
target,no_fragmentation
model,XGBClassifier_no_fragmentation_smotenc_optuna-...
holdout_test_f1_macro,0.882524
holdout_test_accuracy_balanced,0.888636
holdout_test_roc_auc,0.977273
holdout_test_f1,0.829268
holdout_test_accuracy,0.906667
cv_test_f1_macro_median,0.875762
cv_test_accuracy_balanced_median,0.867216
cv_test_roc_auc_median,0.968864


Model saved in ../results/models_modelling4/XGBClassifier_no_fragmentation_smotenc_optuna-opt-mean-smote_default-model


# AdaBoost

In [17]:
estimator = AdaBoostClassifier
target = 'splashing'
estimator_params = {}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-16 02:13:50,556] A new study created in memory with name: smote_study
[I 2025-04-16 02:13:50,854] Trial 0 finished with value: 0.8465841046344754 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.8465841046344754.
[I 2025-04-16 02:13:51,146] Trial 1 finished with value: 0.8522359242923272 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 1 with value: 0.8522359242923272.
[I 2025-04-16 02:13:51,429] Trial 2 finished with value: 0.8507884917610742 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 1 with value: 0.8522359242923272.
[I 2025-04-16 02:13:51,716] Trial 3 finished with value: 0.8275714960645149 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 1 with value: 0.8522359242923272.
[I 2025-04-16 02:13:52,004] Trial 4 finished with value: 0.8442622531338235 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "AdaBoostClassifier"


,0
target,splashing
model,AdaBoostClassifier_splashing_smotenc_optuna-op...
holdout_test_f1_macro,0.781171
holdout_test_accuracy_balanced,0.778935
holdout_test_roc_auc,0.853009
holdout_test_f1,0.845361
holdout_test_accuracy,0.8
cv_test_f1_macro_median,0.781238
cv_test_accuracy_balanced_median,0.80031
cv_test_roc_auc_median,0.890476


Model saved in ../results/models_modelling4/AdaBoostClassifier_splashing_smotenc_optuna-opt-mean-smote_default-model


In [18]:
estimator = AdaBoostClassifier
target = 'no_fragmentation'
estimator_params = {}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-16 02:15:48,522] A new study created in memory with name: smote_study
[I 2025-04-16 02:15:48,820] Trial 0 finished with value: 0.8100472350174143 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.8100472350174143.
[I 2025-04-16 02:15:49,104] Trial 1 finished with value: 0.8327141252068964 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 1 with value: 0.8327141252068964.
[I 2025-04-16 02:15:49,382] Trial 2 finished with value: 0.8221280786372585 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 1 with value: 0.8327141252068964.
[I 2025-04-16 02:15:49,668] Trial 3 finished with value: 0.8084962750878398 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 1 with value: 0.8327141252068964.
[I 2025-04-16 02:15:50,006] Trial 4 finished with value: 0.8299048683403927 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "AdaBoostClassifier"


,0
target,no_fragmentation
model,AdaBoostClassifier_no_fragmentation_smotenc_op...
holdout_test_f1_macro,0.825397
holdout_test_accuracy_balanced,0.852273
holdout_test_roc_auc,0.942273
holdout_test_f1,0.755556
holdout_test_accuracy,0.853333
cv_test_f1_macro_median,0.820946
cv_test_accuracy_balanced_median,0.851648
cv_test_roc_auc_median,0.923077


Model saved in ../results/models_modelling4/AdaBoostClassifier_no_fragmentation_smotenc_optuna-opt-mean-smote_default-model


# Random Forest

In [19]:
estimator = RandomForestClassifier
target = 'splashing'
estimator_params = {}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-16 02:17:44,399] A new study created in memory with name: smote_study
[I 2025-04-16 02:17:44,899] Trial 0 finished with value: 0.863786377358881 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.863786377358881.
[I 2025-04-16 02:17:45,381] Trial 1 finished with value: 0.8665822336614083 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 1 with value: 0.8665822336614083.
[I 2025-04-16 02:17:45,853] Trial 2 finished with value: 0.865402513971099 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 1 with value: 0.8665822336614083.
[I 2025-04-16 02:17:46,344] Trial 3 finished with value: 0.8727387197319347 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 3 with value: 0.8727387197319347.
[I 2025-04-16 02:17:46,828] Trial 4 finished with value: 0.8677002312310957 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best is

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "RandomForestClassifier"


,0
target,splashing
model,RandomForestClassifier_splashing_smotenc_optun...
holdout_test_f1_macro,0.806892
holdout_test_accuracy_balanced,0.799769
holdout_test_roc_auc,0.934028
holdout_test_f1,0.868687
holdout_test_accuracy,0.826667
cv_test_f1_macro_median,0.876935
cv_test_accuracy_balanced_median,0.887302
cv_test_roc_auc_median,0.940402


Model saved in ../results/models_modelling4/RandomForestClassifier_splashing_smotenc_optuna-opt-mean-smote_default-model


In [20]:
estimator = RandomForestClassifier
target = 'no_fragmentation'
estimator_params = {}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-16 02:21:04,797] A new study created in memory with name: smote_study
[I 2025-04-16 02:21:05,310] Trial 0 finished with value: 0.8418425587679417 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.8418425587679417.
[I 2025-04-16 02:21:05,808] Trial 1 finished with value: 0.8243094663104128 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 0 with value: 0.8418425587679417.
[I 2025-04-16 02:21:06,297] Trial 2 finished with value: 0.8275366133875924 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 0 with value: 0.8418425587679417.
[I 2025-04-16 02:21:06,806] Trial 3 finished with value: 0.833939373764748 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 0 with value: 0.8418425587679417.
[I 2025-04-16 02:21:07,299] Trial 4 finished with value: 0.8385083288612777 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best 

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "RandomForestClassifier"


,0
target,no_fragmentation
model,RandomForestClassifier_no_fragmentation_smoten...
holdout_test_f1_macro,0.916089
holdout_test_accuracy_balanced,0.922727
holdout_test_roc_auc,0.983636
holdout_test_f1,0.878049
holdout_test_accuracy,0.933333
cv_test_f1_macro_median,0.875762
cv_test_accuracy_balanced_median,0.867216
cv_test_roc_auc_median,0.948718


Model saved in ../results/models_modelling4/RandomForestClassifier_no_fragmentation_smotenc_optuna-opt-mean-smote_default-model


# LightGBM

In [21]:
estimator = LGBMClassifier
target = 'splashing'
estimator_params = {
    'verbose': -1,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-16 02:24:26,593] A new study created in memory with name: smote_study
[I 2025-04-16 02:24:28,643] Trial 0 finished with value: 0.8781883373901153 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.8781883373901153.
[I 2025-04-16 02:24:30,543] Trial 1 finished with value: 0.8612341097674941 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 0 with value: 0.8781883373901153.
[I 2025-04-16 02:24:32,327] Trial 2 finished with value: 0.8753705038533592 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 0 with value: 0.8781883373901153.
[I 2025-04-16 02:24:34,405] Trial 3 finished with value: 0.8808793523323549 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 3 with value: 0.8808793523323549.
[I 2025-04-16 02:24:36,425] Trial 4 finished with value: 0.8714197607450326 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "LGBMClassifier"


,0
target,splashing
model,LGBMClassifier_splashing_smotenc_optuna-opt-me...
holdout_test_f1_macro,0.836601
holdout_test_accuracy_balanced,0.828704
holdout_test_roc_auc,0.948688
holdout_test_f1,0.888889
holdout_test_accuracy,0.853333
cv_test_f1_macro_median,0.879545
cv_test_accuracy_balanced_median,0.888545
cv_test_roc_auc_median,0.952012


Model saved in ../results/models_modelling4/LGBMClassifier_splashing_smotenc_optuna-opt-mean-smote_default-model


In [22]:
estimator = LGBMClassifier
target = 'no_fragmentation'
estimator_params = {
    'verbose': -1,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-16 02:39:43,748] A new study created in memory with name: smote_study
[I 2025-04-16 02:39:46,904] Trial 0 finished with value: 0.844034830222273 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.844034830222273.
[I 2025-04-16 02:39:49,970] Trial 1 finished with value: 0.8592114748564038 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 1 with value: 0.8592114748564038.
[I 2025-04-16 02:39:52,598] Trial 2 finished with value: 0.8338970611481458 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 1 with value: 0.8592114748564038.
[I 2025-04-16 02:39:55,486] Trial 3 finished with value: 0.8478415427275516 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 1 with value: 0.8592114748564038.
[I 2025-04-16 02:39:58,575] Trial 4 finished with value: 0.8402413822939901 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best i

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "LGBMClassifier"


,0
target,no_fragmentation
model,LGBMClassifier_no_fragmentation_smotenc_optuna...
holdout_test_f1_macro,0.916089
holdout_test_accuracy_balanced,0.922727
holdout_test_roc_auc,0.98
holdout_test_f1,0.878049
holdout_test_accuracy,0.933333
cv_test_f1_macro_median,0.898077
cv_test_accuracy_balanced_median,0.880037
cv_test_roc_auc_median,0.967521


Model saved in ../results/models_modelling4/LGBMClassifier_no_fragmentation_smotenc_optuna-opt-mean-smote_default-model
